## ライブラリのインポート

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import japanize_matplotlib
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

## 万博用のデータ作成

In [2]:
from pipeline.etl.merge_df import load_base_data
from pipeline.features.make_lag_features import make_lag_features
from pipeline.features.make_event_flag import make_event_flag
from pipeline.features.make_station_distance_features import make_station_distance_features

df2 = load_base_data()
df2 = make_lag_features(df2)
df2 = make_event_flag('expo', '../data/raw/Osaka_Events/expo_event_time_series.csv', df2)
df2 = make_event_flag('ir', '../data/raw/Osaka_Events/ir_event_time_series.csv', df2)
df2 = make_station_distance_features(df2)

/Users/shoki_u/ml-training/objective/mlit_property_price_prediction/.venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [3]:
# df2 のシェイプを確認
print(df2.shape)
# df2 のカラムの確認
print(df2.columns.tolist())
# 此花区の行を取り出し
df2[df2['municipality'] == "此花区"].head()

(43804, 46)
['id', 'type', 'price_info_class', 'municipality_code', 'municipality', 'nearest_station', 'layout', 'structure', 'use', 'future_use', 'city_planning', 'renovation', 'transaction_notes', 'transaction_price', 'coverage_ratio', 'floor_area_ratio', 'area_sqm', 'area_no_max_limit_flag', 'building_year', 'building_before_war_flag', 'time_range_write_flag', 'time_no_max_limit_flag', 'time_to_station', 'transaction_year', 'transaction_quarter', 'actual_foreign_guests', 'foreign_male_counts', 'foreign_female_counts', 'total_counts', 'include_foreign_household_counts', 'facility_count', 'price_per_sqm', 'median_price_1year_ago', 'median_price_2year_ago', 'median_price_3year_ago', 'median_price_4year_ago', 'median_price_5year_ago', 'expo_announcement_flag', 'expo_development_start_flag', 'expo_development_completion_flag', 'expo_implementation_flag', 'ir_announcement_flag', 'ir_development_start_flag', 'ir_development_completion_flag', 'ir_implementation_flag', 'distance_to_yumeshima

,id,type,price_info_class,municipality_code,municipality,nearest_station,layout,structure,use,future_use,...,median_price_5year_ago,expo_announcement_flag,expo_development_start_flag,expo_development_completion_flag,expo_implementation_flag,ir_announcement_flag,ir_development_start_flag,ir_development_completion_flag,ir_implementation_flag,distance_to_yumeshima_km
13,8dba9cfb-d79f-431e-84a3-9af963792a4d,中古マンション等,不動産取引価格情報,27104,此花区,安治川口,３ＬＤＫ,ＲＣ,住宅,住宅,...,342857.142857,1,1,0,0,1,0,0,0,5.553492
96,0e65f001-07d5-48b8-93c6-6d88ba1130da,中古マンション等,不動産取引価格情報,27104,此花区,ユニバーサルシティ,NaN,ＲＣ,NaN,住宅,...,271794.871795,1,1,0,0,1,0,0,0,4.833863
97,bdd35f75-3850-4711-80ee-df03f2d6c1ea,中古マンション等,不動産取引価格情報,27104,此花区,ユニバーサルシティ,NaN,ＳＲＣ,NaN,NaN,...,271794.871795,1,1,0,0,1,0,0,0,4.833863
98,f932d068-3099-4429-bedc-773ef7a8280d,中古マンション等,不動産取引価格情報,27104,此花区,西九条,３ＬＤＫ,ＲＣ,住宅,住宅,...,271794.871795,1,1,0,0,1,0,0,0,7.776143
166,ab00cfbc-13f3-4656-95e1-62352867ff93,中古マンション等,不動産取引価格情報,27104,此花区,伝法,３ＬＤＫ,ＳＲＣ,住宅,住宅,...,261538.461538,1,1,0,0,1,0,0,0,7.039255


In [4]:
print([c for c in df2.columns if 'completion' in c or 'announcement' in c or 'implementation' in c or 'start' in c])

['expo_announcement_flag', 'expo_development_start_flag', 'expo_development_completion_flag', 'expo_implementation_flag', 'ir_announcement_flag', 'ir_development_start_flag', 'ir_development_completion_flag', 'ir_implementation_flag']


## IR用の予測データ作成

In [19]:
from pipeline.features.make_df3 import make_df3
df3 = make_df3(df2)

# df3のシェイプ確認
print('【df3のshape】')
print(df3.shape)
print('----' * 10)

【df3のshape】
(43804, 44)
----------------------------------------


In [21]:
# expo_*_flagが全て1になっていることを確認
muni_list = ['此花区', '鶴見区', '住之江区', '淀川区', '福島区']
for muni in muni_list:
    print(f'【{muni}のデータ総数】')
    print(len(df3[df3['municipality'] == muni]))
    print(f'【{muni}のexpo_*_flagの数の確認】')
    print(df3[df3['municipality'] == muni][['expo_announcement_flag',
       'expo_development_start_flag', 'expo_development_completion_flag',
       'expo_implementation_flag']].value_counts())

【此花区のデータ総数】
536
【此花区のexpo_*_flagの数の確認】
expo_announcement_flag  expo_development_start_flag  expo_development_completion_flag  expo_implementation_flag
1                       1                            1                                 1                           536
Name: count, dtype: int64
【鶴見区のデータ総数】
805
【鶴見区のexpo_*_flagの数の確認】
expo_announcement_flag  expo_development_start_flag  expo_development_completion_flag  expo_implementation_flag
1                       0                            0                                 0                           805
Name: count, dtype: int64
【住之江区のデータ総数】
1141
【住之江区のexpo_*_flagの数の確認】
expo_announcement_flag  expo_development_start_flag  expo_development_completion_flag  expo_implementation_flag
0                       1                            1                                 0                           1141
Name: count, dtype: int64
【淀川区のデータ総数】
4287
【淀川区のexpo_*_flagの数の確認】
expo_announcement_flag  expo_development_start_flag  expo_developmen

In [ ]:
# ir_*_flagの確認
df3.pivot_table(values=['ir_announcement_flag',
       'ir_development_start_flag', 'ir_development_completion_flag',
       'ir_implementation_flag'], index='municipality', columns='transaction_year', aggfunc='median')

ir_announcement_flag                      \
transaction_year                 2026 2027 2028 2029 2030   
municipality                                                
中央区                               0.0  0.0  0.0  0.0  0.0   
住之江区                              0.0  0.0  0.0  0.0  0.0   
住吉区                               0.0  0.0  0.0  0.0  0.0   
北区                                0.0  0.0  0.0  0.0  0.0   
城東区                               0.0  0.0  0.0  0.0  0.0   
大正区                               0.0  0.0  0.0  0.0  0.0   
天王寺区                              0.0  0.0  0.0  0.0  0.0   
平野区                               0.0  0.0  0.0  0.0  0.0   
旭区                                0.0  0.0  0.0  0.0  0.0   
東住吉区                              0.0  0.0  0.0  0.0  0.0   
東成区                               0.0  0.0  0.0  0.0  0.0   
東淀川区                              0.0  0.0  0.0  0.0  0.0   
此花区                               1.0  1.0  1.0  1.0  1.0   
浪速区                               0.0  0.0  0.0  0.0  0.0   
淀川区                               0.0  0.0  0.0  0.0  0.0   
港区                                0.0  0.0  0.0  0.0  0.0   
生野区                               0.0  0.0  0.0  0.0  0.0   
福島区                               0.0  0.0  0.0  0.0  0.0   
西区                                0.0  0.0  0.0  0.0  0.0   
西成区                               0.0  0.0  0.0  0.0  0.0   
西淀川区                              0.0  0.0  0.0  0.0  0.0   
都島区                               0.0  0.0  0.0  0.0  0.0   
阿倍野区                              0.0  0.0  0.0  0.0  0.0   
鶴見区                               0.0  0.0  0.0  0.0  0.0   

                 ir_development_completion_flag                      \
transaction_year                           2026 2027 2028 2029 2030   
municipality                                                          
中央区                                         0.0  0.0  0.0  0.0  0.0   
住之江区                                        0.0  0.0  0.0  0.0  0.0   
住吉区                                         0.0  0.0  0.0  0.0  0.0   
北区                                          0.0  0.0  0.0  0.0  0.0   
城東区                                         0.0  0.0  0.0  0.0  0.0   
大正区                                         0.0  0.0  0.0  0.0  0.0   
天王寺区                                        0.0  0.0  0.0  0.0  0.0   
平野区                                         0.0  0.0  0.0  0.0  0.0   
旭区                                          0.0  0.0  0.0  0.0  0.0   
東住吉区                                        0.0  0.0  0.0  0.0  0.0   
東成区                                         0.0  0.0  0.0  0.0  0.0   
東淀川区                                        0.0  0.0  0.0  0.0  0.0   
此花区                                         0.0  0.0  0.0  0.0  1.0   
浪速区                                         0.0  0.0  0.0  0.0  0.0   
淀川区                                         0.0  0.0  0.0  0.0  0.0   
港区                                          0.0  0.0  0.0  0.0  0.0   
生野区                                         0.0  0.0  0.0  0.0  0.0   
福島区                                         0.0  0.0  0.0  0.0  0.0   
西区                                          0.0  0.0  0.0  0.0  0.0   
西成区                                         0.0  0.0  0.0  0.0  0.0   
西淀川区                                        0.0  0.0  0.0  0.0  0.0   
都島区                                         0.0  0.0  0.0  0.0  0.0   
阿倍野区                                        0.0  0.0  0.0  0.0  0.0   
鶴見区                                         0.0  0.0  0.0  0.0  0.0   

                 ir_development_start_flag                      \
transaction_year                      2026 2027 2028 2029 2030   
municipality                                                     
中央区                                    0.0  0.0  0.0  0.0  0.0   
住之江区                                   0.0  0.0  0.0  0.0  0.0   
住吉区                                    0.0  0.0  0.0  0.0  0.0   
北区                                     0.0  0.0  0.0  0